In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import r2_score, mean_squared_error

In [44]:
df_train = pd.read_csv("Data/Regularization/train.csv")
df_test = pd.read_csv("Data/Regularization/test.csv")

In [45]:
print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

Train shape: (320, 12)
Test shape: (80, 12)


In [46]:
train_clean = df_train.drop(columns=["ID"]) if "ID" in df_train.columns else df_train.copy()
test_clean = df_test.drop(columns=["ID"]) if "ID" in df_test.columns else df_test.copy()

In [47]:
# Categorical columns: Gender, Student, Married, Ethnicity
print("\nUnique values in categorical columns:")
for col in ["Gender", "Student", "Married", "Ethnicity"]:
    if col in train_clean.columns:
        print(f"{col}: {train_clean[col].unique()}")


Unique values in categorical columns:
Gender: ['Female' ' Male']
Student: ['No' 'Yes']
Married: ['No' 'Yes']
Ethnicity: ['Caucasian' 'African American' 'Asian']


In [48]:
# Create dummy variables
train_encoded = pd.get_dummies(train_clean, columns=["Gender", "Student", "Married", "Ethnicity"], drop_first=True, dtype=float)
test_encoded = pd.get_dummies(test_clean, columns=["Gender", "Student", "Married", "Ethnicity"], drop_first=True, dtype=float)

In [49]:
# Align columns to ensure both have the same dummies
train_encoded, test_encoded = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0.0)

In [50]:
# Split into X and y
X_train = train_encoded.drop(columns=["Balance"])
y_train = train_encoded["Balance"]
X_test = test_encoded.drop(columns=["Balance"])
y_test = test_encoded["Balance"]

In [51]:
numeric_cols = ["Income", "Limit", "Rating", "Cards", "Age", "Education"]
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

In [52]:
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [53]:
ols_base = LinearRegression()
ols_base.fit(X_train_scaled, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [54]:
y_train_pred = ols_base.predict(X_train_scaled)
y_test_pred = ols_base.predict(X_test_scaled)

In [55]:
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print(f"OLS Baseline - Train R2: {train_r2:.4f}, Test R2: {test_r2:.4f}")
print(f"OLS Baseline - Train RMSE: {train_rmse:.4f}, Test RMSE: {test_rmse:.4f}")

OLS Baseline - Train R2: 0.9546, Test R2: 0.9560
OLS Baseline - Train RMSE: 98.4282, Test RMSE: 93.3608


In [56]:
# VIF Calculation
def get_vif(df):
    vifs = {}
    for col in df.columns:
        y = df[col]
        X = df.drop(columns=[col])
        lr = LinearRegression()
        lr.fit(X, y)
        r2 = lr.score(X, y)
        vifs[col] = 1.0 / (1.0 - r2) if r2 < 1.0 else float('inf')
    return pd.DataFrame({"Feature": vifs.keys(), "VIF": vifs.values()})

vif_df = get_vif(X_train_scaled)
print("\nBaseline OLS VIF Scores:")
print(vif_df.to_string(index=False))


Baseline OLS VIF Scores:
            Feature        VIF
             Income   2.915317
              Limit 242.277667
             Rating 245.025791
              Cards   1.451930
                Age   1.068264
          Education   1.030240
      Gender_Female   1.007451
        Student_Yes   1.052633
        Married_Yes   1.049157
    Ethnicity_Asian   1.525768
Ethnicity_Caucasian   1.505123


In [57]:
X_train_reduced = X_train_scaled.drop(columns=["Rating"])
X_test_reduced = X_test_scaled.drop(columns=["Rating"])

# Recalculate VIF
vif_reduced = get_vif(X_train_reduced)
print("VIF after dropping 'Rating':")
print(vif_reduced.to_string(index=False))

VIF after dropping 'Rating':
            Feature      VIF
             Income 2.891419
              Limit 2.792567
              Cards 1.013596
                Age 1.068263
          Education 1.025006
      Gender_Female 1.006325
        Student_Yes 1.038520
        Married_Yes 1.040044
    Ethnicity_Asian 1.520765
Ethnicity_Caucasian 1.504535


In [58]:
# Fit OLS on reduced set
ols_reduced = LinearRegression()
ols_reduced.fit(X_train_reduced, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [59]:
y_train_pred_red = ols_reduced.predict(X_train_reduced)
y_test_pred_red = ols_reduced.predict(X_test_reduced)

In [60]:
red_train_r2 = r2_score(y_train, y_train_pred_red)
red_test_r2 = r2_score(y_test, y_test_pred_red)
red_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_red))
red_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_red))

print(f"Reduced OLS - Train R2: {red_train_r2:.4f}, Test R2: {red_test_r2:.4f}")
print(f"Reduced OLS - Train RMSE: {red_train_rmse:.4f}, Test RMSE: {red_test_rmse:.4f}")

Reduced OLS - Train R2: 0.9541, Test R2: 0.9549
Reduced OLS - Train RMSE: 98.9910, Test RMSE: 94.5241


In [61]:
alphas = np.logspace(-4, 4, 200)
ridge_cv = RidgeCV(alphas=alphas, cv=5, scoring='neg_mean_squared_error')
ridge_cv.fit(X_train_scaled, y_train)

print(f"Optimal Ridge alpha: {ridge_cv.alpha_:.4f}")
ridge_best = ridge_cv

y_train_pred_ridge = ridge_best.predict(X_train_scaled)
y_test_pred_ridge = ridge_best.predict(X_test_scaled)

ridge_train_r2 = r2_score(y_train, y_train_pred_ridge)
ridge_test_r2 = r2_score(y_test, y_test_pred_ridge)
ridge_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_ridge))
ridge_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_ridge))

print(f"Ridge Model - Train R2: {ridge_train_r2:.4f}, Test R2: {ridge_test_r2:.4f}")
print(f"Ridge Model - Train RMSE: {ridge_train_rmse:.4f}, Test RMSE: {ridge_test_rmse:.4f}")


Optimal Ridge alpha: 0.1035
Ridge Model - Train R2: 0.9546, Test R2: 0.9561
Ridge Model - Train RMSE: 98.4386, Test RMSE: 93.2351


In [62]:
lasso_cv = LassoCV(cv=5, max_iter=10000, random_state=42)
lasso_cv.fit(X_train_scaled, y_train)

print(f"Optimal Lasso alpha: {lasso_cv.alpha_:.4f}")
lasso_best = lasso_cv

y_train_pred_lasso = lasso_best.predict(X_train_scaled)
y_test_pred_lasso = lasso_best.predict(X_test_scaled)

lasso_train_r2 = r2_score(y_train, y_train_pred_lasso)
lasso_test_r2 = r2_score(y_test, y_test_pred_lasso)
lasso_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_lasso))
lasso_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_lasso))

print(f"Lasso Model - Train R2: {lasso_train_r2:.4f}, Test R2: {lasso_test_r2:.4f}")
print(f"Lasso Model - Train RMSE: {lasso_train_rmse:.4f}, Test RMSE: {lasso_test_rmse:.4f}")

Optimal Lasso alpha: 0.4555
Lasso Model - Train R2: 0.9545, Test R2: 0.9561
Lasso Model - Train RMSE: 98.4868, Test RMSE: 93.2139


In [63]:
print("\n=== 6. Model Comparison & Summary ===")
summary_df = pd.DataFrame({
    "Model": ["Baseline OLS", "Reduced OLS (Drop Rating)", "Ridge Regression", "Lasso Regression"],
    "Train R2": [train_r2, red_train_r2, ridge_train_r2, lasso_train_r2],
    "Test R2": [test_r2, red_test_r2, ridge_test_r2, lasso_test_r2],
    "Train RMSE": [train_rmse, red_train_rmse, ridge_train_rmse, lasso_train_rmse],
    "Test RMSE": [test_rmse, red_test_rmse, ridge_test_rmse, lasso_test_rmse]
})
print(summary_df.to_string(index=False))

# Compare coefficients
coef_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Baseline OLS": ols_base.coef_
})

# Add Reduced OLS coefficients (inserting 0 or NaN for Rating)
reduced_coefs = []
for col in X_train.columns:
    if col == "Rating":
        reduced_coefs.append(np.nan)
    else:
        # Find index in reduced feature list
        idx = X_train_reduced.columns.get_loc(col)
        reduced_coefs.append(ols_reduced.coef_[idx])
coef_df["Reduced OLS"] = reduced_coefs
coef_df["Ridge"] = ridge_best.coef_
coef_df["Lasso"] = lasso_best.coef_

print("\nCoefficient Comparison Across Models:")
print(coef_df.round(3).to_string(index=False))



=== 6. Model Comparison & Summary ===
                    Model  Train R2  Test R2  Train RMSE  Test RMSE
             Baseline OLS  0.954587 0.955981   98.428193  93.360764
Reduced OLS (Drop Rating)  0.954066 0.954877   98.991016  94.524144
         Ridge Regression  0.954578 0.956099   98.438579  93.235056
         Lasso Regression  0.954533 0.956119   98.486786  93.213884

Coefficient Comparison Across Models:
            Feature  Baseline OLS  Reduced OLS    Ridge    Lasso
             Income      -284.887     -283.258 -284.451 -282.665
              Limit       459.943      623.068  439.177  466.476
             Rating       165.001          NaN  185.359  156.107
              Cards        23.799       30.778   22.935   23.630
                Age        -8.715       -8.725   -8.778   -8.451
          Education        -5.661       -6.424   -5.521   -5.167
      Gender_Female        -7.097       -6.389   -7.129   -4.931
        Student_Yes       429.184      433.100  427.229  424.8